# 第18篇｜阶段大实战：电商用户行为分析完整项目

> 这是「数据分析从入门到精通」系列的第 18 篇，也是 Pandas 阶段的收官实战。前 17 篇学了那么多工具，今天全部用上——从一份真实的电商用户行为数据出发，完整跑通数据分析的全流程。

---

嗨，我是小荷～

第一阶段要收官了！18 篇下来，我们学了 NumPy 的矩阵运算、Pandas 的数据清洗、多表合并、分组聚合……但"知道"和"会用"是两回事。

今天这篇，我们做一个**完整的电商用户行为分析项目**：

- 数据读取 + 检查
- 数据清洗（缺失值、重复值、类型转换）
- 多表合并
- 分组聚合分析
- 透视表制作
- 关键指标计算
- 输出分析报告

整个流程走完，你就有了第一份可以写进简历的数据分析作品了 🎉

> 萧何说：「运筹帷幄之中，决胜千里之外」。他的底气，来自于对所有信息的掌握。数据分析师的底气，来自于能把一堆原始数据变成清晰的洞察。今天，就是我们练这个底气的时候。

---

## 一、项目背景与数据说明

**业务背景**：某电商平台希望了解用户的消费行为，回答以下问题：

1. 总体业务情况如何？（GMV、订单量、客单价）
2. 哪些城市贡献了最多销售额？
3. 不同品类的销售情况差异大吗？
4. 高价值用户（RFM模型）是谁？
5. 各会员等级的消费差异是否显著？

**数据表说明**：
- `orders`：订单表（order_id, user_id, product_id, amount, order_date, status）
- `users`：用户表（user_id, name, city, register_date, level）
- `products`：商品表（product_id, name, category, price_tier）

---

## 二、完整代码

### Step 1：数据准备

先把数据准备好，这是分析的第一步：


In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

np.random.seed(2024)

# ── 生成模拟数据 ──
n_orders = 2000
n_users = 300
n_products = 50

# 用户表
users = pd.DataFrame({
    'user_id': range(1, n_users + 1),
    'name': [f'用户{i:03d}' for i in range(1, n_users + 1)],
    'city': np.random.choice(['北京', '上海', '广州', '深圳', '杭州', '成都', '武汉'], n_users,
                              p=[0.2, 0.2, 0.15, 0.15, 0.1, 0.1, 0.1]),
    'register_date': pd.date_range('2022-01-01', periods=n_users, freq='36h'),
    'level': np.random.choice(['普通', '银卡', '金卡', '钻石'], n_users, p=[0.45, 0.30, 0.15, 0.10])
})

# 商品表
categories = ['数码', '服装', '食品', '美妆', '图书', '运动', '家居']
products = pd.DataFrame({
    'product_id': [f'P{i:03d}' for i in range(1, n_products + 1)],
    'product_name': [f'商品{i:03d}' for i in range(1, n_products + 1)],
    'category': np.random.choice(categories, n_products),
    'price_tier': np.random.choice(['高', '中', '低'], n_products, p=[0.2, 0.5, 0.3])
})

# 订单表（故意引入一些脏数据）
orders = pd.DataFrame({
    'order_id': range(10001, 10001 + n_orders),
    'user_id': np.random.choice(range(1, n_users + 1), n_orders),
    'product_id': np.random.choice([f'P{i:03d}' for i in range(1, n_products + 1)], n_orders),
    'amount': np.random.choice(
        list(np.random.randint(50, 3000, n_orders - 20)) + [None] * 10 + [-999] * 5 + [99999] * 5, n_orders
    ),
    'order_date': pd.date_range('2024-01-01', periods=n_orders, freq='4h'),
    'status': np.random.choice(['已完成', '已完成', '已完成', '已退款', '待支付'], n_orders)
})

# 制造一些重复订单
orders = pd.concat([orders, orders.sample(30)], ignore_index=True)

print("原始数据形状：")
print(f"  订单表: {orders.shape}")
print(f"  用户表: {users.shape}")
print(f"  商品表: {products.shape}")


原始数据形状：
  订单表: (2030, 6)
  用户表: (300, 5)
  商品表: (50, 4)


---

### Step 2：数据检查与清洗

数据拿到手先别急着分析，检查一下质量：


In [5]:
print("\n" + "="*50)
print("【Step 2】数据检查与清洗")
print("="*50)

# ── 检查缺失值 ──
print("\n缺失值统计：")
print(orders.isnull().sum())

# ── 检查重复值 ──
dup_count = orders.duplicated(subset='order_id').sum()
print(f"\n重复订单数：{dup_count}")

# ── 删除重复行 ──
orders_clean = orders.drop_duplicates(subset='order_id')
print(f"去重后订单数：{len(orders_clean)}")

# ── 处理缺失值（amount 缺失的删除）──
orders_clean = orders_clean.dropna(subset=['amount'])
print(f"删除 amount 缺失后：{len(orders_clean)}")

# ── 处理异常值（amount < 0 或 > 50000 视为异常）──
orders_clean['amount'] = pd.to_numeric(orders_clean['amount'], errors='coerce')
outlier_mask = (orders_clean['amount'] < 0) | (orders_clean['amount'] > 50000)
print(f"异常金额订单数：{outlier_mask.sum()}")
orders_clean = orders_clean[~outlier_mask]

# ── 只保留"已完成"订单用于分析 ──
orders_analysis = orders_clean[orders_clean['status'] == '已完成'].copy()
print(f"\n有效已完成订单数：{len(orders_analysis)}")

# ── 时间字段处理 ──
orders_analysis['order_date'] = pd.to_datetime(orders_analysis['order_date'])
orders_analysis['order_month'] = orders_analysis['order_date'].dt.to_period('M').astype(str)
orders_analysis['order_week']  = orders_analysis['order_date'].dt.isocalendar().week



【Step 2】数据检查与清洗

缺失值统计：
order_id       0
user_id        0
product_id     0
amount        10
order_date     0
status         0
dtype: int64

重复订单数：30
去重后订单数：2000
删除 amount 缺失后：1990
异常金额订单数：14

有效已完成订单数：1219


---

### Step 3：多表合并

多张表合到一起，才能做完整分析：


In [6]:
print("\n" + "="*50)
print("【Step 3】多表合并")
print("="*50)

# 关联用户信息
df = pd.merge(orders_analysis, users, on='user_id', how='left')
# 关联商品信息
df = pd.merge(df, products[['product_id', 'category', 'price_tier']], on='product_id', how='left')

print(f"合并后数据形状：{df.shape}")
print(f"合并后列名：{list(df.columns)}")

# 检查关联后的缺失情况
print(f"\n合并后 city 缺失：{df['city'].isnull().sum()}")
print(f"合并后 category 缺失：{df['category'].isnull().sum()}")



【Step 3】多表合并
合并后数据形状：(1219, 14)
合并后列名：['order_id', 'user_id', 'product_id', 'amount', 'order_date', 'status', 'order_month', 'order_week', 'name', 'city', 'register_date', 'level', 'category', 'price_tier']

合并后 city 缺失：0
合并后 category 缺失：0


---

### Step 4：业务分析

数据准备好了，来回答业务问题：


In [7]:
print("\n" + "="*50)
print("【Step 4】业务分析")
print("="*50)

# ── 指标1：总体概况 ──
total_gmv    = df['amount'].sum()
total_orders = len(df)
avg_order    = df['amount'].mean()
unique_users = df['user_id'].nunique()

print(f"""
📊 总体业务概况
  总 GMV：      ¥{total_gmv:,.0f}
  总订单数：     {total_orders:,}
  平均客单价：   ¥{avg_order:.1f}
  活跃用户数：   {unique_users}
""")

# ── 指标2：各城市销售额 TOP 5 ──
city_sales = df.groupby('city').agg(
    总销售额=('amount', 'sum'),
    订单数=('order_id', 'count'),
    平均客单价=('amount', 'mean'),
    用户数=('user_id', 'nunique')
).round(1).sort_values('总销售额', ascending=False)

print("🏙️ 各城市销售情况：")
print(city_sales)

# ── 指标3：各品类销售情况 ──
cat_sales = df.groupby('category').agg(
    总销售额=('amount', 'sum'),
    订单占比=('order_id', 'count')
).round(1)
cat_sales['订单占比'] = (cat_sales['订单占比'] / len(df) * 100).round(1)
cat_sales = cat_sales.sort_values('总销售额', ascending=False)

print("\n🛒 各品类销售情况：")
print(cat_sales)

# ── 指标4：月度销售趋势 ──
monthly = df.groupby('order_month')['amount'].agg(['sum', 'count', 'mean']).round(1)
monthly.columns = ['月销售额', '订单数', '客单价']
print("\n📅 月度销售趋势：")
print(monthly)

# ── 指标5：会员等级消费差异 ──
level_analysis = df.groupby('level').agg(
    用户数=('user_id', 'nunique'),
    总消费=('amount', 'sum'),
    平均客单价=('amount', 'mean'),
    人均消费=('amount', lambda x: x.sum() / max(df[df['level'] == x.name]['user_id'].nunique(), 1))
).round(1)
print("\n👑 各会员等级消费分析：")
print(level_analysis)



【Step 4】业务分析

📊 总体业务概况
  总 GMV：      ¥1,870,394
  总订单数：     1,219
  平均客单价：   ¥1534.4
  活跃用户数：   293

🏙️ 各城市销售情况：
        总销售额  订单数   平均客单价  用户数
city                          
北京    345757  221  1564.5   50
深圳    340911  226  1508.5   44
上海    324979  211  1540.2   60
广州    281981  180  1566.6   45
杭州    222165  143  1553.6   36
成都    196306  133  1476.0   33
武汉    158295  105  1507.6   25

🛒 各品类销售情况：
            总销售额  订单占比
category              
运动        521868  28.5
数码        358337  19.6
家居        241227  12.8
美妆        219063  11.5
食品        202015  10.4
图书        165446   8.9
服装        162438   8.3

📅 月度销售趋势：
               月销售额  订单数     客单价
order_month                     
2024-01      166955  110  1517.8
2024-02      165009  116  1422.5
2024-03      146847   99  1483.3
2024-04      184941  119  1554.1
2024-05      191255  116  1648.8
2024-06      173344  110  1575.9
2024-07      178802  120  1490.0
2024-08      186312  110  1693.7
2024-09      181634  116  1565.8
2024-10      1

---

### Step 5：RFM 用户价值分析

RFM 模型是用户价值分析的经典方法：


In [8]:
print("\n" + "="*50)
print("【Step 5】RFM 用户价值分析")
print("="*50)

# R = Recency（最近一次购买距今天数）
# F = Frequency（购买频次）
# M = Monetary（总消费金额）

analysis_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('user_id').agg(
    R=('order_date', lambda x: (analysis_date - x.max()).days),
    F=('order_id', 'count'),
    M=('amount', 'sum')
).round(1)

# 按分位数打分（1~4分，4最好）
rfm['R_score'] = pd.qcut(rfm['R'], q=4, labels=[4, 3, 2, 1])  # R越小越好，所以倒序
rfm['F_score'] = pd.qcut(rfm['F'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_score'] = pd.qcut(rfm['M'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

# 综合打标签
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

def rfm_label(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    total = r + f + m
    if total >= 10:
        return '⭐ 高价值用户'
    elif total >= 8:
        return '🔵 潜力用户'
    elif r >= 3:
        return '🟡 新用户'
    else:
        return '🔴 流失风险'

rfm['user_type'] = rfm.apply(rfm_label, axis=1)

print("RFM 用户分层结果：")
print(rfm['user_type'].value_counts())
print(f"\n总用户数：{len(rfm)}")
print("\n高价值用户 RFM 均值：")
print(rfm[rfm['user_type'] == '⭐ 高价值用户'][['R', 'F', 'M']].mean().round(1))



【Step 5】RFM 用户价值分析
RFM 用户分层结果：
user_type
🔴 流失风险     104
⭐ 高价值用户     78
🔵 潜力用户      69
🟡 新用户       42
Name: count, dtype: int64

总用户数：293

高价值用户 RFM 均值：
R       35.1
F        6.5
M    10572.7
dtype: float64


---

### Step 6：输出透视报表

最后输出一张透视报表，分析结果一目了然：


In [9]:
print("\n" + "="*50)
print("【Step 6】输出透视报表")
print("="*50)

# 城市 × 品类 销售额报表
pivot_report = pd.pivot_table(
    df,
    values='amount',
    index='city',
    columns='category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='合计'
).round(0).astype(int)

print("\n📊 城市 × 品类 销售额（元）：")
print(pivot_report)

# 会员等级 × 品类 客单价报表
pivot_report2 = pd.pivot_table(
    df,
    values='amount',
    index='level',
    columns='category',
    aggfunc='mean',
    fill_value=0
).round(1)

print("\n📊 会员等级 × 品类 平均客单价：")
print(pivot_report2)



【Step 6】输出透视报表

📊 城市 × 品类 销售额（元）：
category      图书      家居      数码      服装      美妆      运动      食品       合计
city                                                                     
上海         14296   40074   65854   30556   37104  101190   35905   324979
北京         31857   37234   75500   22988   49984   97361   30833   345757
广州         37454   32848   41177   20757   33619   74830   41296   281981
成都         11443   37618   42929   19690   24388   41959   18279   196306
杭州         25952   30916   35786   24089   17172   58926   29324   222165
武汉         22499   22923   29561    7860   17607   44625   13220   158295
深圳         21945   39614   67530   36498   39189  102977   33158   340911
合计        165446  241227  358337  162438  219063  521868  202015  1870394

📊 会员等级 × 品类 平均客单价：
category      图书      家居      数码      服装      美妆      运动      食品
level                                                           
普通        1312.5  1513.6  1487.1  1650.7  1484.1  1495.6  1764.4
金卡        

---

### Step 7：生成分析结论

分析做完，总结一下关键发现：


In [10]:
print("\n" + "="*50)
print("【Step 7】分析结论")
print("="*50)

top_city = city_sales.index[0]
top_category = cat_sales.index[0]

print(f"""
📌 核心结论：

1. 总体情况
   - 分析期内共完成 {total_orders:,} 笔有效订单，总 GMV ¥{total_gmv:,.0f}
   - 平均客单价 ¥{avg_order:.0f}，活跃用户 {unique_users} 人

2. 地域分布
   - {top_city} 贡献销售额最高，是核心市场

3. 品类表现
   - {top_category} 品类销售额最高，是主力品类

4. 用户价值
   - 高价值用户（RFM ≥ 10分）购买频次高、消费金额大、近期活跃
   - 建议优先维护高价值用户，对"流失风险"用户发放召回优惠

5. 会员等级
   - 钻石/金卡用户客单价显著高于普通用户
   - 建议通过积分激励让普通/银卡用户向金卡升级
""")

print("✅ 第一阶段大实战完成！")



【Step 7】分析结论

📌 核心结论：

1. 总体情况
   - 分析期内共完成 1,219 笔有效订单，总 GMV ¥1,870,394
   - 平均客单价 ¥1534，活跃用户 293 人

2. 地域分布
   - 北京 贡献销售额最高，是核心市场

3. 品类表现
   - 运动 品类销售额最高，是主力品类

4. 用户价值
   - 高价值用户（RFM ≥ 10分）购买频次高、消费金额大、近期活跃
   - 建议优先维护高价值用户，对"流失风险"用户发放召回优惠

5. 会员等级
   - 钻石/金卡用户客单价显著高于普通用户
   - 建议通过积分激励让普通/银卡用户向金卡升级

✅ 第一阶段大实战完成！


---

## 三、🎯 项目复盘：用到了哪些技能？

| 技能 | 对应篇章 |
|------|---------|
| 数据读取、检查 | 第08、09篇 |
| 缺失值/重复值处理 | 第11篇 |
| 数据类型转换（日期） | 第12篇 |
| 行列筛选 | 第10篇 |
| 多表合并 | 第15篇 |
| 分组聚合 | 第14篇 |
| apply 自定义函数 | 第17篇 |
| 数据透视表 | 第16篇 |
| NumPy 数值计算 | 第02~06篇 |

这18篇，一篇都没有白学 💪

---

## 四、📁 作品集建议

把这个项目整理成一份报告，附上：

1. **数据说明**：数据来源、字段解释
2. **分析过程**：完整清洗步骤说明
3. **核心结论**：5 条可执行的业务洞察
4. **代码**：可运行的 Jupyter Notebook

这就是一份完整的数据分析作品了，找实习/找工作时可以直接展示 👌

---

## 下期预告

> **第 19 篇：Matplotlib 基础 — 画布 / 坐标轴 / 图层**
>
> 第一阶段圆满收官！下一个阶段是数据可视化，我们要让数据「开口说话」——用图表把分析结论展示出来。下篇从 Matplotlib 最核心的画布和坐标轴开始。

---

*跟着小荷，数据分析路上不迷路～*
*（第一阶段结束！萧何同学：运筹帷幄，决胜千里！我们做到了 🎉）*
